In [4]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [5]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://www.nature.com/articles/s41590-021-00922-4"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41590-021-00922-4/MediaObjects/41590_2021_922_MOESM2_ESM.xlsx"

# don't include in header
table_name = "41590_2021_922_MOESM2_ESM.xlsx"
table_name = "paper_degs.xlsx" # local name # using this for paper_degs

## Read in file

In [6]:
excel = pd.read_excel(table_name, sheet_name= "all markers", skiprows = 1)

df = excel

ValueError: Worksheet named 'all markers' not found

In [4]:
df.head()

,Unnamed: 0,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene
0,CXCL14,0.0,3.699825,0.426,0.082,0.0,ASPC,CXCL14
1,NEGR1,0.0,3.668332,0.867,0.247,0.0,ASPC,NEGR1
2,DCN,0.0,3.565394,0.960,0.506,0.0,ASPC,DCN
3,LAMA2,0.0,3.417902,0.829,0.266,0.0,ASPC,LAMA2
4,APOD,0.0,3.383653,0.493,0.120,0.0,ASPC,APOD


## Unfiltered

In [5]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"cluster": "cell_type_label", "gene": "feature_name"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None

unfiltered_df.drop(['Unnamed: 0', 'p_val', 'avg_log2FC', 'pct.1', 'pct.2', 'p_val_adj'], axis=1, inplace=True) 

result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'ASPC',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'CXCL14',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'ASPC',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'CXCL14',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'ASPC',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'NEGR1',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'ASPC',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'NEGR1',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': '

## Save Unfiltered

In [6]:
with open("evidence_unfiltered.json", "w") as f:
    json.dump(result, f, indent = 4) 

## Perform the filtering

In [7]:
col_pval = "p_val_adj"
col_lfc = "avg_log2FC"
col_gene = "gene"
col_ct = "cluster" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [8]:
"""
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20
"""

min_mean = 0
max_pval = 1
min_lfc = 0
max_gene_shares = 100
max_per_celltype = 100

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [9]:
markers

,Unnamed: 0,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene
16961,MME2,1.957379e-24,2.601825,0.377,0.134,6.172202e-20,neutrophil,MME
17357,HLA-B6,9.655946e-03,1.049600,0.390,0.392,1.000000e+00,neutrophil,HLA-B
17140,FKBP54,2.264181e-04,1.773418,0.397,0.352,1.000000e+00,neutrophil,FKBP5
17013,CCND36,8.739571e-12,1.998003,0.397,0.230,2.755849e-07,neutrophil,CCND3
17180,TMCC12,7.088423e-04,1.681569,0.411,0.414,1.000000e+00,neutrophil,TMCC1
...,...,...,...,...,...,...,...,...
3107,CD361,0.000000e+00,1.757112,0.982,0.426,0.000000e+00,adipocyte,CD36
3713,FTX2,0.000000e+00,0.467243,0.984,0.811,0.000000e+00,adipocyte,FTX
3078,EBF12,0.000000e+00,2.098168,0.991,0.557,0.000000e+00,adipocyte,EBF1
3140,NEAT1,0.000000e+00,1.573039,0.997,0.864,0.000000e+00,adipocyte,NEAT1


In [10]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
neutrophil        0.56716
t_cell            0.61417
nk_cell           0.64111
SMC               0.65865
mast_cell         0.66189
ASPC              0.66816
pericyte          0.67154
b_cell            0.69602
endometrium       0.69805
dendritic_cell    0.70000
endothelial       0.71003
monocyte          0.71417
LEC               0.73262
macrophage        0.76192
mesothelium       0.79866
adipocyte         0.89949
Name: pct.1, dtype: float64

In [11]:
markers[col_ct].value_counts()

cluster
neutrophil        100
t_cell            100
nk_cell           100
mast_cell         100
SMC               100
ASPC              100
pericyte          100
endometrium       100
b_cell            100
dendritic_cell    100
endothelial       100
monocyte          100
LEC               100
mesothelium       100
macrophage        100
adipocyte         100
Name: count, dtype: int64

## Find Ensembl ID


In [21]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [22]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [23]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [12]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = None

result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "filtered", "source_id": "deg"}})

result

[{'cell_type_label': 'neutrophil',
  'gene': 'MME',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'HLA-B',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'FKBP5',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'CCND3',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'TMCC1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'ELMO1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'LUCAT1',
  'organism': 'homo_sap

## Save evidence

In [13]:
with open("evidence.json", "w") as f:
    json.dump(result, f, indent = 4) 